# EDL Full Experiment (auto-parallel, per-(dataset,model) hyperparams)

10-fold cross-validation comparison of 8 LDL models across the datasets
listed in `DATASETS`, parallelised via `loky`. The pool config is
auto-detected by `edl_workers.auto_pool_config()`:

- **GPU mode** if ≥ 2 CUDA GPUs are visible to `nvidia-smi` — one worker
  pinned per GPU.
- **CPU mode** otherwise — a small number of fat CPU workers (≈
  `cpu_count // 4`).

The training loop lives in `edl_workers.py` next to this notebook so loky
subprocesses can `import edl_workers` cleanly. TensorFlow is imported
lazily inside each worker after `CUDA_VISIBLE_DEVICES` is pinned.

For each (model, dataset) pair we record six distributional metrics
(`chebyshev`, `clark`, `canberra`, `kl_divergence`, `cosine`,
`intersection`) across 10 folds, plus per-sample uncertainty for the
evidential / SNEFY models.

Hyperparameters are configured per-(dataset, model) in the **Hyperparams**
section below. Anything you don't override falls back to `DEFAULT_HP`.


In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import importlib
import sys
import multiprocessing as mp
from collections import defaultdict
from concurrent.futures import as_completed
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from loky import get_reusable_executor

# Make edl_workers importable whether the kernel started in `demo/` or in the
# project root.
_demo_dir = Path.cwd() if (Path.cwd() / 'edl_workers.py').exists() else Path.cwd() / 'demo'
if str(_demo_dir) not in sys.path:
    sys.path.insert(0, str(_demo_dir))

import pyldl.utils
import pyldl.algorithms
importlib.reload(pyldl.utils)
importlib.reload(pyldl.algorithms)

from pyldl.utils import load_dataset
from edl_workers import (
    init_worker, run_one_fold, auto_pool_config,
    MODEL_NAMES, METRICS, DEFAULT_HP,
)


E0000 00:00:1777403620.708517 2327698 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777403620.712596 2327698 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777403620.725282 2327698 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777403620.725305 2327698 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777403620.725307 2327698 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777403620.725308 2327698 computation_placer.cc:177] computation placer already registered. Please check linka

## Datasets and CV settings


In [2]:
DATASETS     = ['SJAFFE', 'SBU_3DFE', 'Movie']
N_SPLITS     = 10
RANDOM_STATE = 0


## Hyperparams

`DEFAULT_HP` (defined in `edl_workers.py`) supplies the fallback for any
key that isn't overridden:

```
n_hidden       64       # dense width for the encoder/MLP
learning_rate  1e-3     # AdamW; set to None to use each model's own optimizer
weight_decay   1e-4     # AdamW
dropout_rate   0.0      # inserted between hidden and output if > 0
epochs         2500     # upper bound; early stopping cuts it short
batch_size     None     # None → each model's own default
val_fraction   0.1      # of the training fold; held out as inner-val for early-stop
patience       50       # LDLEarlyStopping; set None to disable
minimum        100      # LDLEarlyStopping; minimum epochs before early-stop can fire
max_iterations 50       # SA_BFGS only
```

Override per (dataset, model) in `PER_DATASET_HP` below. Anything you
don't list uses `DEFAULT_HP` as-is.


In [3]:
PER_DATASET_HP = {
    'SJAFFE': {
        'AA_BP':                    {'patience': 100, 'minimum': 100, 'n_hidden': 64},
        'Duo_LDL':                  {'patience': 100, 'minimum': 100, 'n_hidden': 64},
        'EDL_LDL (loglikelihood)':  {'patience': 100, 'minimum': 100, 'n_hidden': 64},
        'EDL_LDL (bayes_mse)':      {'patience': 100, 'minimum': 100, 'n_hidden': 64},
        'BEDL_LDL (loglikelihood)': {'patience': 100, 'minimum': 100, 'n_hidden': 64},
        'BEDL_LDL (bayes_mse)':     {'patience': 100, 'minimum': 100, 'n_hidden': 64},
        'SNEFY_LDL':                {'patience': 100, 'minimum': 100, 'n_hidden': 64},
        'SA_BFGS':                  {},
    },
    'SBU_3DFE': {
        'AA_BP':                    {'patience': 20, 'minimum': 20, 'n_hidden': 32, 'weight_decay': 1e-4, 'batch_size': 256},
        'Duo_LDL':                  {'patience': 20, 'minimum': 20, 'n_hidden': 32, 'weight_decay': 1e-4, 'batch_size': 256},
        'EDL_LDL (loglikelihood)':  {'patience': 20, 'minimum': 20, 'n_hidden': 32, 'weight_decay': 1e-4, 'batch_size': 256},
        'EDL_LDL (bayes_mse)':      {'patience': 20, 'minimum': 20, 'n_hidden': 32, 'weight_decay': 1e-4, 'batch_size': 256},
        'BEDL_LDL (loglikelihood)': {'patience': 20, 'minimum': 20, 'n_hidden': 32, 'weight_decay': 1e-4, 'batch_size': 256},
        'BEDL_LDL (bayes_mse)':     {'patience': 20, 'minimum': 20, 'n_hidden': 32, 'weight_decay': 1e-4, 'batch_size': 256},
        'SNEFY_LDL':                {'patience': 100, 'minimum': 100, 'n_hidden': 32, 'weight_decay': 1e-4, 'batch_size': 256},
        'SA_BFGS':                  {},
    },
    'Natural_Scene': {
        'AA_BP':                    {},
        'Duo_LDL':                  {},
        'EDL_LDL (loglikelihood)':  {},
        'EDL_LDL (bayes_mse)':      {},
        'BEDL_LDL (loglikelihood)': {},
        'BEDL_LDL (bayes_mse)':     {},
        'SNEFY_LDL':                {},
        'SA_BFGS':                  {},
    },
    'Movie': {
        'AA_BP':                    {'patience': 20, 'minimum': 20, 'n_hidden': 4, 'weight_decay': 1e-3, 'batch_size': 512},
        'Duo_LDL':                  {'patience': 20, 'minimum': 20, 'n_hidden': 4, 'weight_decay': 1e-3, 'batch_size': 512},
        'EDL_LDL (loglikelihood)':  {'patience': 20, 'minimum': 20, 'n_hidden': 4, 'weight_decay': 1e-3, 'batch_size': 512},
        'EDL_LDL (bayes_mse)':      {'patience': 20, 'minimum': 20, 'n_hidden': 4, 'weight_decay': 1e-3, 'batch_size': 512},
        'BEDL_LDL (loglikelihood)': {'patience': 20, 'minimum': 20, 'n_hidden': 4, 'weight_decay': 1e-3, 'batch_size': 512},
        'BEDL_LDL (bayes_mse)':     {'patience': 20, 'minimum': 20, 'n_hidden': 4, 'weight_decay': 1e-3, 'batch_size': 512},
        'SNEFY_LDL':                {'patience': 20, 'minimum': 20, 'n_hidden': 4, 'weight_decay': 1e-3, 'batch_size': 512},
        'SA_BFGS':                  {},
    },
}


def resolve_hp(dataset, model_name):
    """Merge DEFAULT_HP with the per-(dataset, model) overrides."""
    hp = dict(DEFAULT_HP)
    hp.update(PER_DATASET_HP.get(dataset, {}).get(model_name, {}))
    return hp


## Pool configuration (auto-detected)


In [16]:
# Set POOL_CFG to a dict to override; leave None for auto-detect.
POOL_CFG = None
if POOL_CFG is None:
    POOL_CFG = auto_pool_config()

GPU_IDS   = POOL_CFG['gpu_ids']
N_WORKERS = POOL_CFG['n_workers']
INTRA     = POOL_CFG['intra_op_threads']
INTER     = POOL_CFG['inter_op_threads']
MODE      = POOL_CFG['mode']

print(f'mode      : {MODE}')
print(f'workers   : {N_WORKERS}')
print(f'gpu_ids   : {GPU_IDS if GPU_IDS else "(CPU only)"}')
print(f'tf threads: intra={INTRA}, inter={INTER}')


mode      : GPU
workers   : 3
gpu_ids   : [0, 1, 2]
tf threads: intra=1, inter=1


## Build the worker pool


In [5]:
mgr = mp.Manager()
gpu_queue = mgr.Queue()

slots = list(GPU_IDS) if GPU_IDS else [None] * N_WORKERS
assert len(slots) == N_WORKERS, 'one queue slot per worker'
for g in slots:
    gpu_queue.put(g)

executor = get_reusable_executor(
    max_workers=N_WORKERS,
    initializer=init_worker,
    initargs=(gpu_queue, INTRA, INTER),
    reuse=False,
)
print(f'pool ready: {N_WORKERS} workers, slots={slots}')


pool ready: 3 workers, slots=[0, 1, 2]


## Sanity check: what do the workers actually see?

`gpu_devices` should match what you expect for the mode:

- **GPU mode**: each worker has a different `CUDA_VISIBLE_DEVICES` and
  reports a single GPU device.
- **CPU mode**: every worker reports `CUDA_VISIBLE_DEVICES = '-1'` and
  an empty `gpu_devices` list.


In [6]:
def _check():
    import os, tensorflow as tf
    return {
        'pid': os.getpid(),
        'CUDA_VISIBLE_DEVICES': os.environ.get('CUDA_VISIBLE_DEVICES'),
        'gpu_devices': [d.name for d in tf.config.list_physical_devices('GPU')],
        'intra_threads': tf.config.threading.get_intra_op_parallelism_threads(),
        'inter_threads': tf.config.threading.get_inter_op_parallelism_threads(),
    }

checks = [executor.submit(_check) for _ in range(N_WORKERS * 4)]
df = pd.DataFrame([c.result() for c in checks]).drop_duplicates(subset='pid').reset_index(drop=True)
print(f'{len(df)} unique workers (expected {N_WORKERS})')
df


E0000 00:00:1777403834.981793 2328440 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777403834.985811 2328440 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
E0000 00:00:1777403834.991258 2328442 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777403834.995352 2328442 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777403835.007320 2328440 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777403835.007341 2328440 computation_placer.cc:177] computation placer already registered. Please check li

3 unique workers (expected 3)


,pid,CUDA_VISIBLE_DEVICES,gpu_devices,intra_threads,inter_threads
0,2328442,1,[/physical_device:GPU:0],1,1
1,2328441,2,[/physical_device:GPU:0],1,1
2,2328440,0,[/physical_device:GPU:0],1,1


## Build the job list

Each job carries the resolved `hp` dict for its (dataset, model)
combination, so workers don't need to know about `PER_DATASET_HP`.


In [7]:
jobs = []
for dataset_name in DATASETS:
    X, D = load_dataset(dataset_name, dir='dataset')
    print(f'{dataset_name}: {X.shape[0]} samples, {X.shape[1]} features, {D.shape[1]} labels')
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
        Xtr, Xte = X[train_idx], X[test_idx]
        Dtr, Dte = D[train_idx], D[test_idx]
        for model_name in MODEL_NAMES:
            hp = resolve_hp(dataset_name, model_name)
            jobs.append((dataset_name, model_name, fold_idx, Xtr, Dtr, Xte, Dte, hp))

total = len(jobs)
print(f'queued {total} jobs ({len(DATASETS)} datasets × {N_SPLITS} folds × {len(MODEL_NAMES)} models)')


SJAFFE: 213 samples, 243 features, 6 labels
SBU_3DFE: 2500 samples, 243 features, 6 labels
Movie: 7755 samples, 1869 features, 5 labels
queued 240 jobs (3 datasets × 10 folds × 8 models)


### Inspect the resolved hyperparams

Confirm the config you'll actually run with — defaults plus your
per-(dataset, model) overrides.


In [8]:
hp_rows = []
for ds in DATASETS:
    for m in MODEL_NAMES:
        hp = resolve_hp(ds, m)
        hp_rows.append({'dataset': ds, 'model': m, **hp})
hp_table = pd.DataFrame(hp_rows).set_index(['dataset', 'model'])
hp_table


n_hidden  learning_rate  weight_decay  \
dataset  model                                                             
SJAFFE   EDL_LDL (loglikelihood)         64          0.001        0.0001   
         EDL_LDL (bayes_mse)             64          0.001        0.0001   
         BEDL_LDL (loglikelihood)        64          0.001        0.0001   
         BEDL_LDL (bayes_mse)            64          0.001        0.0001   
         Duo_LDL                         64          0.001        0.0001   
         AA_BP                           64          0.001        0.0001   
         SNEFY_LDL                       64          0.001        0.0001   
         SA_BFGS                         64          0.001        0.0001   
SBU_3DFE EDL_LDL (loglikelihood)         32          0.001        0.0001   
         EDL_LDL (bayes_mse)             32          0.001        0.0001   
         BEDL_LDL (loglikelihood)        32          0.001        0.0001   
         BEDL_LDL (bayes_mse)            32          0.001        0.0001   
         Duo_LDL                         32          0.001        0.0001   
         AA_BP                           32          0.001        0.0001   
         SNEFY_LDL                       32          0.001        0.0001   
         SA_BFGS                         64          0.001        0.0001   
Movie    EDL_LDL (loglikelihood)          4          0.001        0.0010   
         EDL_LDL (bayes_mse)              4          0.001        0.0010   
         BEDL_LDL (loglikelihood)         4          0.001        0.0010   
         BEDL_LDL (bayes_mse)             4          0.001        0.0010   
         Duo_LDL                          4          0.001        0.0010   
         AA_BP                            4          0.001        0.0010   
         SNEFY_LDL                        4          0.001        0.0010   
         SA_BFGS                         64          0.001        0.0001   

                                   dropout_rate  epochs  batch_size  \
dataset  model                                                        
SJAFFE   EDL_LDL (loglikelihood)            0.2    2500         NaN   
         EDL_LDL (bayes_mse)                0.2    2500         NaN   
         BEDL_LDL (loglikelihood)           0.2    2500         NaN   
         BEDL_LDL (bayes_mse)               0.2    2500         NaN   
         Duo_LDL                            0.2    2500         NaN   
         AA_BP                              0.2    2500         NaN   
         SNEFY_LDL                          0.2    2500         NaN   
         SA_BFGS                            0.2    2500         NaN   
SBU_3DFE EDL_LDL (loglikelihood)            0.2    2500       256.0   
         EDL_LDL (bayes_mse)                0.2    2500       256.0   
         BEDL_LDL (loglikelihood)           0.2    2500       256.0   
         BEDL_LDL (bayes_mse)               0.2    2500       256.0   
         Duo_LDL                            0.2    2500       256.0   
         AA_BP                              0.2    2500       256.0   
         SNEFY_LDL                          0.2    2500       256.0   
         SA_BFGS                            0.2    2500         NaN   
Movie    EDL_LDL (loglikelihood)            0.2    2500       512.0   
         EDL_LDL (bayes_mse)                0.2    2500       512.0   
         BEDL_LDL (loglikelihood)           0.2    2500       512.0   
         BEDL_LDL (bayes_mse)               0.2    2500       512.0   
         Duo_LDL                            0.2    2500       512.0   
         AA_BP                              0.2    2500       512.0   
         SNEFY_LDL                          0.2    2500       512.0   
         SA_BFGS                            0.2    2500         NaN   

                                   val_fraction  patience  minimum  \
dataset  model                                                       
SJAFFE   EDL_LDL (loglikelihood)            0.1       100      100   
 

## Submit + collect

Submission is non-blocking; results stream back via `as_completed`. Per-fold
failures are caught and logged but don't stop the run.


In [9]:
futures = {
    executor.submit(run_one_fold, ds, m, fi, Xtr, Dtr, Xte, Dte, hp): (ds, m, fi)
    for (ds, m, fi, Xtr, Dtr, Xte, Dte, hp) in jobs
}

raw_results = []
failures = []
for i, fut in enumerate(as_completed(futures), start=1):
    ds, m, fi = futures[fut]
    try:
        raw_results.append(fut.result())
        status = 'ok'
    except Exception as e:
        failures.append((ds, m, fi, repr(e)))
        status = f'FAILED ({type(e).__name__}: {e})'
    print(f'[{i:4d}/{total}] {ds:12s} | fold {fi:2d} | {m:30s} {status}')

print(f'\ndone: {len(raw_results)} ok, {len(failures)} failed')


E0000 00:00:1777403890.276325 2328545 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777403890.280838 2328545 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777403890.305322 2328545 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777403890.305437 2328545 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777403890.305468 2328545 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777403890.305492 2328545 computation_placer.cc:177] computation placer already registered. Please check linka

[   1/240] SJAFFE       | fold  1 | EDL_LDL (loglikelihood)        ok
[   2/240] SJAFFE       | fold  1 | BEDL_LDL (loglikelihood)       ok
[   3/240] SJAFFE       | fold  1 | EDL_LDL (bayes_mse)            ok
[   4/240] SJAFFE       | fold  1 | BEDL_LDL (bayes_mse)           ok
[   5/240] SJAFFE       | fold  1 | AA_BP                          ok
[   6/240] SJAFFE       | fold  1 | SA_BFGS                        ok
[   7/240] SJAFFE       | fold  1 | Duo_LDL                        ok
[   8/240] SJAFFE       | fold  2 | EDL_LDL (loglikelihood)        ok
[   9/240] SJAFFE       | fold  2 | BEDL_LDL (loglikelihood)       ok
[  10/240] SJAFFE       | fold  2 | EDL_LDL (bayes_mse)            ok
[  11/240] SJAFFE       | fold  2 | BEDL_LDL (bayes_mse)           ok
[  12/240] SJAFFE       | fold  1 | SNEFY_LDL                      ok
[  13/240] SJAFFE       | fold  2 | AA_BP                          ok
[  14/240] SJAFFE       | fold  2 | SA_BFGS                        ok
[  15/240] SJAFFE   

E0000 00:00:1777405353.011761 2450436 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777405353.017785 2450436 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777405353.030198 2450436 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777405353.030295 2450436 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777405353.030331 2450436 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777405353.030360 2450436 computation_placer.cc:177] computation placer already registered. Please check linka

[ 162/240] SBU_3DFE     | fold 10 | BEDL_LDL (loglikelihood)       ok
[ 163/240] Movie        | fold  1 | EDL_LDL (bayes_mse)            ok
[ 164/240] Movie        | fold  1 | BEDL_LDL (bayes_mse)           ok
[ 165/240] Movie        | fold  1 | Duo_LDL                        ok
[ 166/240] Movie        | fold  1 | SA_BFGS                        ok
[ 167/240] Movie        | fold  1 | SNEFY_LDL                      FAILED (TypeError: object of type 'NoneType' has no len())
[ 168/240] Movie        | fold  1 | AA_BP                          ok


E0000 00:00:1777405453.668672 2453799 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777405453.673103 2453799 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777405453.685499 2453799 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777405453.685591 2453799 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777405453.685621 2453799 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777405453.685645 2453799 computation_placer.cc:177] computation placer already registered. Please check linka

...

[ 231/240] Movie        | fold  9 | Duo_LDL                        ok
[ 232/240] Movie        | fold  9 | AA_BP                          ok
[ 233/240] Movie        | fold 10 | EDL_LDL (loglikelihood)        ok


E0000 00:00:1777406412.432291 2486667 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777406412.436661 2486667 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777406412.448916 2486667 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777406412.449010 2486667 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777406412.449041 2486667 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777406412.449066 2486667 computation_placer.cc:177] computation placer already registered. Please check linka

[ 234/240] Movie        | fold 10 | BEDL_LDL (loglikelihood)       ok


/home/dcs01/.local/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
I0000 00:00:1777406423.845090 2486667 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 7650 MB memory:  -> device: 0, name: NVIDIA H100 80GB HBM3 MIG 1g.10gb, pci bus id: 0000:0c:00.0, compute capability: 9.0


[ 235/240] Movie        | fold 10 | EDL_LDL (bayes_mse)            ok
[ 236/240] Movie        | fold 10 | Duo_LDL                        ok
[ 237/240] Movie        | fold 10 | BEDL_LDL (bayes_mse)           ok
[ 238/240] Movie        | fold 10 | SA_BFGS                        ok
[ 239/240] Movie        | fold 10 | SNEFY_LDL                      FAILED (TypeError: object of type 'NoneType' has no len())
[ 240/240] Movie        | fold 10 | AA_BP                          ok

done: 230 ok, 10 failed


### Worker-state diagnostic

If the submit cell hangs, run this in a separate cell to see if any
workers have died. Healthy state = all workers `sleeping`/`running` with
non-trivial `rss`. Dead PIDs in the `dead` list = OOM-killed or crashed,
in which case rebuild the pool and resubmit only `missing` jobs.


In [ ]:
import psutil

worker_pids = list(executor._processes.keys())
alive, dead = [], []
for pid in worker_pids:
    try:
        p = psutil.Process(pid)
        if p.is_running() and p.status() != psutil.STATUS_ZOMBIE:
            alive.append((pid, p.status(), p.memory_info().rss / 1e9, p.cpu_percent(interval=0.5)))
        else:
            dead.append(pid)
    except psutil.NoSuchProcess:
        dead.append(pid)

print(f'alive workers ({len(alive)}/{N_WORKERS}):')
for pid, status, rss_gb, cpu in alive:
    print(f'  pid={pid}  status={status}  rss={rss_gb:.1f}GB  cpu={cpu:.0f}%')
print(f'dead workers: {dead}')
print(f'\nresults so far: {len(raw_results)} / {total}')
print(f'completed futures: {sum(f.done() for f in futures)}')
print(f'pending futures:   {sum(not f.done() for f in futures)}')


## Bucket results into per-model DataFrames


In [10]:
buckets = defaultdict(list)
for r in raw_results:
    buckets[(r['dataset'], r['model'])].append(r['scores'])

per_model_results = {key: pd.DataFrame(rows) for key, rows in buckets.items()}
print(f'{len(per_model_results)} (dataset, model) combinations have results')


23 (dataset, model) combinations have results


## Per-model fold tables


In [11]:
if per_model_results:
    first_key = next(iter(per_model_results))
    print(f'showing: {first_key}')
    display(per_model_results[first_key])
else:
    print('no results yet — run the submit cell first')


showing: ('SJAFFE', 'EDL_LDL (loglikelihood)')


,chebyshev,clark,canberra,kl_divergence,cosine,intersection,mean_uncertainty,uncertainty_calibration
0,0.138662,0.474765,1.022165,0.091973,0.913105,0.823461,0.589772,-0.189159
1,0.128364,0.457179,0.968833,0.079271,0.924308,0.834559,0.572139,0.555775
2,0.129258,0.438752,0.905251,0.078405,0.925533,0.844238,0.592153,-0.588933
3,0.114620,0.390878,0.813435,0.063263,0.938972,0.860259,0.346719,-0.112987
4,0.103143,0.430306,0.896292,0.065382,0.939984,0.852348,0.589863,-0.018182
5,0.124777,0.425657,0.901544,0.084081,0.923120,0.845267,0.168534,0.228571
6,0.119544,0.421769,0.887102,0.073684,0.931104,0.848716,0.589529,-0.149351
7,0.110321,0.413030,0.852734,0.064215,0.940094,0.857899,0.168397,-0.215584
8,0.114374,0.423489,0.887972,0.070165,0.933828,0.849907,0.587663,0.128571
9,0.113394,0.388088,0.831635,0.067054,0.935970,0.856179,0.584553,0.040260


## Combined summary — mean ± std across folds


In [12]:
def summarize(df):
    out = {}
    for col in df.columns:
        out[f'{col}_mean'] = df[col].mean()
        out[f'{col}_std']  = df[col].std()
    return out


summary_rows = []
for (dataset_name, model_name), df in per_model_results.items():
    if df.empty:
        continue
    row = {'dataset': dataset_name, 'model': model_name, **summarize(df)}
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).set_index(['dataset', 'model'])
summary


chebyshev_mean  chebyshev_std  clark_mean  \
dataset  model                                                                 
SJAFFE   EDL_LDL (loglikelihood)         0.119646       0.010582    0.426391   
         BEDL_LDL (loglikelihood)        0.117727       0.011637    0.418165   
         EDL_LDL (bayes_mse)             0.120396       0.009920    0.430536   
         BEDL_LDL (bayes_mse)            0.120148       0.010074    0.429345   
         AA_BP                           0.119290       0.011377    0.426552   
         SA_BFGS                         0.087746       0.008730    0.326173   
         Duo_LDL                         0.119538       0.011076    0.430547   
         SNEFY_LDL                       0.144438       0.022902    0.554543   
SBU_3DFE EDL_LDL (loglikelihood)         0.137306       0.002745    0.410570   
         EDL_LDL (bayes_mse)             0.136437       0.003132    0.412252   
         Duo_LDL                         0.134806       0.002396    0.417340   
         AA_BP                           0.134981       0.002632    0.416889   
         BEDL_LDL (bayes_mse)            0.135274       0.002866    0.414751   
         SA_BFGS                         0.128959       0.002345    0.403880   
         SNEFY_LDL                       0.187100       0.040790    0.717544   
         BEDL_LDL (loglikelihood)        0.128580       0.005435    0.386926   
Movie    EDL_LDL (loglikelihood)         0.116226       0.002676    0.525218   
         BEDL_LDL (loglikelihood)        0.118012       0.002841    0.532703   
         EDL_LDL (bayes_mse)             0.124058       0.004956    0.560412   
         BEDL_LDL (bayes_mse)            0.118094       0.003039    0.535716   
         Duo_LDL                         0.119404       0.003971    0.545104   
         SA_BFGS                         0.145060       0.003879    0.586648   
         AA_BP                           0.115334       0.003300    0.523317   

                                   clark_std  canberra_mean  canberra_std  \
dataset  model                                                              
SJAFFE   EDL_LDL (loglikelihood)    0.026652       0.896696      0.061742   
         BEDL_LDL (loglikelihood)   0.033509       0.876625      0.079071   
         EDL_LDL (bayes_mse)        0.025678       0.900584      0.059238   
         BEDL_LDL (bayes_mse)       0.027264       0.896823      0.065307   
         AA_BP                      0.027539       0.889118      0.061249   
         SA_BFGS                    0.027843       0.665279      0.053841   
         Duo_LDL                    0.028214       0.896047      0.060517   
         SNEFY_LDL                  0.112102       1.157216      0.232641   
SBU_3DFE EDL_LDL (loglikelihood)    0.004402       0.896695      0.007651   
         EDL_LDL (bayes_mse)        0.004507       0.898070      0.008817   
         Duo_LDL                    0.004285       0.903614      0.007362   
         AA_BP                      0.003586       0.903325      0.006202   
         BEDL_LDL (bayes_mse)       0.004350       0.900090      0.009177   
         SA_BFGS                    0.006497       0.863641      0.011503   
         SNEFY_LDL                  0.207385       1.451091      0.383958   
         BEDL_LDL (loglikelihood)   0.013746       0.841078      0.030613   
Movie    EDL_LDL (loglikelihood)    0.011939       1.002495      0.021059   
         BEDL_LDL (loglikelihood)   0.011205       1.016737      0.020579   
         EDL_LDL (bayes_mse)        0.020337       1.065178      0.041605   
         BEDL_LDL (bayes_mse)       0.011458       1.016957      0.021985   
         Duo_LDL                    0.016733       1.036006      0.032592   
         SA_BFGS                    0.009972       1.119303      0.017762   
         AA_BP                      0.013723       0.999038      0.025506   

                                   kl_divergence_mean  kl_divergence_std  \
dataset  model                  

### Compact view: `mean ± std` per metric


In [13]:
def fmt(mean, std):
    if pd.isna(mean):
        return ''
    return f'{mean:.4f} ± {std:.4f}'


compact_rows = []
for (dataset_name, model_name), df in per_model_results.items():
    if df.empty:
        continue
    row = {'dataset': dataset_name, 'model': model_name}
    for col in df.columns:
        row[col] = fmt(df[col].mean(), df[col].std())
    compact_rows.append(row)

compact = pd.DataFrame(compact_rows).set_index(['dataset', 'model'])
compact


chebyshev            clark  \
dataset  model                                                        
SJAFFE   EDL_LDL (loglikelihood)   0.1196 ± 0.0106  0.4264 ± 0.0267   
         BEDL_LDL (loglikelihood)  0.1177 ± 0.0116  0.4182 ± 0.0335   
         EDL_LDL (bayes_mse)       0.1204 ± 0.0099  0.4305 ± 0.0257   
         BEDL_LDL (bayes_mse)      0.1201 ± 0.0101  0.4293 ± 0.0273   
         AA_BP                     0.1193 ± 0.0114  0.4266 ± 0.0275   
         SA_BFGS                   0.0877 ± 0.0087  0.3262 ± 0.0278   
         Duo_LDL                   0.1195 ± 0.0111  0.4305 ± 0.0282   
         SNEFY_LDL                 0.1444 ± 0.0229  0.5545 ± 0.1121   
SBU_3DFE EDL_LDL (loglikelihood)   0.1373 ± 0.0027  0.4106 ± 0.0044   
         EDL_LDL (bayes_mse)       0.1364 ± 0.0031  0.4123 ± 0.0045   
         Duo_LDL                   0.1348 ± 0.0024  0.4173 ± 0.0043   
         AA_BP                     0.1350 ± 0.0026  0.4169 ± 0.0036   
         BEDL_LDL (bayes_mse)      0.1353 ± 0.0029  0.4148 ± 0.0043   
         SA_BFGS                   0.1290 ± 0.0023  0.4039 ± 0.0065   
         SNEFY_LDL                 0.1871 ± 0.0408  0.7175 ± 0.2074   
         BEDL_LDL (loglikelihood)  0.1286 ± 0.0054  0.3869 ± 0.0137   
Movie    EDL_LDL (loglikelihood)   0.1162 ± 0.0027  0.5252 ± 0.0119   
         BEDL_LDL (loglikelihood)  0.1180 ± 0.0028  0.5327 ± 0.0112   
         EDL_LDL (bayes_mse)       0.1241 ± 0.0050  0.5604 ± 0.0203   
         BEDL_LDL (bayes_mse)      0.1181 ± 0.0030  0.5357 ± 0.0115   
         Duo_LDL                   0.1194 ± 0.0040  0.5451 ± 0.0167   
         SA_BFGS                   0.1451 ± 0.0039  0.5866 ± 0.0100   
         AA_BP                     0.1153 ± 0.0033  0.5233 ± 0.0137   

                                          canberra    kl_divergence  \
dataset  model                                                        
SJAFFE   EDL_LDL (loglikelihood)   0.8967 ± 0.0617  0.0737 ± 0.0095   
         BEDL_LDL (loglikelihood)  0.8766 ± 0.0791  0.0713 ± 0.0101   
         EDL_LDL (bayes_mse)       0.9006 ± 0.0592  0.0740 ± 0.0084   
         BEDL_LDL (bayes_mse)      0.8968 ± 0.0653  0.0740 ± 0.0090   
         AA_BP                     0.8891 ± 0.0612  0.0737 ± 0.0110   
         SA_BFGS                   0.6653 ± 0.0538  0.0432 ± 0.0090   
         Duo_LDL                   0.8960 ± 0.0605  0.0741 ± 0.0096   
         SNEFY_LDL                 1.1572 ± 0.2326  0.1274 ± 0.0469   
SBU_3DFE EDL_LDL (loglikelihood)   0.8967 ± 0.0077  0.0837 ± 0.0016   
         EDL_LDL (bayes_mse)       0.8981 ± 0.0088  0.0837 ± 0.0022   
         Duo_LDL                   0.9036 ± 0.0074  0.0828 ± 0.0016   
         AA_BP                     0.9033 ± 0.0062  0.0828 ± 0.0016   
         BEDL_LDL (bayes_mse)      0.9001 ± 0.0092  0.0828 ± 0.0019   
         SA_BFGS                   0.8636 ± 0.0115  0.0759 ± 0.0020   
         SNEFY_LDL                 1.4511 ± 0.3840  0.2394 ± 0.1413   
         BEDL_LDL (loglikelihood)  0.8411 ± 0.0306  0.0738 ± 0.0059   
Movie    EDL_LDL (loglikelihood)   1.0025 ± 0.0211  0.0995 ± 0.0036   
         BEDL_LDL (loglikelihood)  1.0167 ± 0.0206  0.1024 ± 0.0044   
         EDL_LDL (bayes_mse)       1.0652 ± 0.0416  0.1113 ± 0.0094   
         BEDL_LDL (bayes_mse)      1.0170 ± 0.0220  0.1024 ± 0.0048   
         Duo_LDL                   1.0360 ± 0.0326  0.1138 ± 0.0105   
         SA_BFGS                   1.1193 ± 0.0178  0.1331 ± 0.0045   
         AA_BP                     0.9990 ± 0.0255  0.0992 ± 0.0046   

                                            cosine     intersection  \
dataset  model                                                        
SJAFFE   EDL_LDL (loglikelihood)   0.9306 ± 0.0089  0.8473 ± 0.0112   
         BEDL_LDL (loglikelihood)  0.9328 ± 0.0099  0.8506 ± 0.0138   
         EDL_LDL (bayes_mse)       0.9302 ± 0.0078  0.8466 ± 0.0104   
         BEDL_LDL (bayes_mse)      0.9303 ± 0.0085  0.8472 ± 0.0117   
         AA_BP                     0.9306 ± 0.0102  0.8484 ± 0.

## Uncertainty results (EDL_LDL, BEDL_LDL, SNEFY_LDL only)

- `mean_uncertainty` — average per-sample uncertainty on test (model-specific
  scale; lower = more confident).
- `uncertainty_calibration` — Spearman ρ between per-sample uncertainty and
  per-sample KL divergence error. Higher = uncertainty better predicts error.


In [14]:
uncertainty_models = {
    'EDL_LDL (loglikelihood)', 'EDL_LDL (bayes_mse)',
    'BEDL_LDL (loglikelihood)', 'BEDL_LDL (bayes_mse)',
    'SNEFY_LDL',
}

uncertainty_rows = []
for (dataset_name, model_name), df in per_model_results.items():
    if model_name not in uncertainty_models or df.empty:
        continue
    if 'mean_uncertainty' not in df.columns:
        continue
    uncertainty_rows.append({
        'dataset': dataset_name,
        'model': model_name,
        'mean_uncertainty':        fmt(df['mean_uncertainty'].mean(),        df['mean_uncertainty'].std()),
        'uncertainty_calibration': fmt(df['uncertainty_calibration'].mean(), df['uncertainty_calibration'].std()),
    })

uncertainty_summary = pd.DataFrame(uncertainty_rows).set_index(['dataset', 'model'])
uncertainty_summary


mean_uncertainty uncertainty_calibration
dataset  model                                                            
SJAFFE   EDL_LDL (loglikelihood)   0.4789 ± 0.1800        -0.0321 ± 0.3043
         BEDL_LDL (loglikelihood)  0.3322 ± 0.1362         0.1666 ± 0.3212
         EDL_LDL (bayes_mse)       0.4593 ± 0.1325         0.1177 ± 0.2179
         BEDL_LDL (bayes_mse)      0.2815 ± 0.1673        -0.0971 ± 0.2446
         SNEFY_LDL                 0.3008 ± 0.2558         0.0905 ± 0.2896
SBU_3DFE EDL_LDL (loglikelihood)   0.3616 ± 0.1906        -0.1077 ± 0.1973
         EDL_LDL (bayes_mse)       0.3101 ± 0.2700        -0.2155 ± 0.0475
         BEDL_LDL (bayes_mse)      0.0309 ± 0.0370        -0.2082 ± 0.0813
         SNEFY_LDL                 0.2411 ± 0.1107        -0.1442 ± 0.1340
         BEDL_LDL (loglikelihood)  0.1652 ± 0.0285         0.2097 ± 0.2320
Movie    EDL_LDL (loglikelihood)   0.2835 ± 0.0164         0.2016 ± 0.0397
         BEDL_LDL (loglikelihood)  0.2365 ± 0.0071         0.1926 ± 0.0243
         EDL_LDL (bayes_mse)       0.1016 ± 0.0693        -0.0210 ± 0.0609
         BEDL_LDL (bayes_mse)      0.0176 ± 0.0360         0.0618 ± 0.0694

## Shut down the pool


In [15]:
executor.shutdown(wait=True, kill_workers=True)
